# QELM Tomography with Free/Interacting Fermion Reservoir

The goal of this notebook is to begin experimenting with performing full quantum state tomography using both a free and interacting fermion model for both the target and reservoir. Ideally, measurements will be restricted to site occupation measures, and the aim is to reproduce something similar to the model implemented on Equal1 chips

## 1. Imports

In [ ]:
from qsim.dynamics import HamiltonianGenerator, ExponentialPropagator
from qsim.state import DensityMatrix
from qsim.lin_alg import Operator

from qres.qelm import QELM

import numpy as np
import scipy

## 2. Reservoir Configuration

In [ ]:
n_target = 2
n_reservoir = 4

min_energy = 0
max_energy = 1

min_tunnel = 0.5
max_tunnel = 1.5

In [ ]:
def freeFermionH(n_target, n_reservoir):
    dim = n_target + n_reservoir
    H = np.triu(np.random.uniform(min_tunnel, max_tunnel, size=(dim, dim)), k=1)
    H += H.T
    for i in range(dim):
        H[i][i] = np.random.uniform(min_energy, max_energy)
    return Operator(H)

In [ ]:
H = freeFermionH(n_target, n_reservoir)

In [ ]:
eigs, U = np.linalg.eigh(H.matrix)
U = Operator(U)

In [ ]:
import numpy as np

def random_free_fermion(n_target, n_reservoir):

    # random unitary
    X = np.random.randn(n_target,n_target) + 1j*np.random.randn(n_target,n_target)
    Q, _ = np.linalg.qr(X)

    # random occupations
    occ = np.random.rand(2)

    C = Q @ np.diag(occ) @ Q.conj().T
    C_full = np.zeros((n_target + n_reservoir, n_target + n_reservoir)).astype(complex)
    C_full[:n_target, :n_target] = C
    return DensityMatrix(C_full)

state = random_free_fermion(n_target, n_reservoir)

print(state.matrix)

[[0.84320644+2.77555756e-17j 0.26727711-2.07751070e-01j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j]
 [0.26727711+2.07751070e-01j 0.14962269+0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j]
 [0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j]
 [0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j]
 [0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j]
 [0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.        +0.0

In [ ]:
entanglement_entropy(np.array([[0,0],[0,0]]), [0,1])

np.float64(5.726199798841585e-11)

In [ ]:
import numpy as np

def entanglement_entropy(C, subsystem):
    """
    Compute entanglement entropy of subsystem from correlation matrix.

    C : full correlation matrix
    subsystem : list of site indices in subsystem
    """

    C_A = C[np.ix_(subsystem, subsystem)]

    # eigenvalues of restricted correlation matrix
    nu = np.linalg.eigvalsh(C_A)

    # numerical stability
    eps = 1e-12
    nu = np.clip(nu, eps, 1-eps)

    S = -np.sum(nu*np.log(nu) + (1-nu)*np.log(1-nu))

    return S

## 3. Training Dataset

In [ ]:
x_train[0].matrix

array([[0.27738709+1.38777878e-17j, 0.21935143+5.76594126e-02j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j],
       [0.21935143-5.76594126e-02j, 0.39605738+1.73472348e-18j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j],
       [0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j],
       [0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j],
       [0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j]])

In [ ]:
n_train = 10
n_test = 100

x_train = [random_free_fermion(n_target, n_reservoir) for i in range(n_train)]
x_test = [random_free_fermion(n_target, n_reservoir) for i in range(n_test)]

y_train = [[x.matrix[0][0].real, x.matrix[1][1].real, x.matrix[0][1].real, x.matrix[0][1].imag] for x in x_train]
y_test = [[x.matrix[0][0].real, x.matrix[1][1].real, x.matrix[0][1].real, x.matrix[0][1].imag] for x in x_test]

## 4. Transform

In [ ]:
U = Operator(scipy.linalg.expm(-1j * H.matrix))
d_train = np.array([np.diag((U @ rho @ U.hConj()).matrix)[n_target:] for rho in x_train]).real
d_test = np.array([np.diag((U @ rho @ U.hConj()).matrix)[n_target:] for rho in x_test]).real

In [ ]:
d_train

array([[0.04421457, 0.00829649, 0.11789976, 0.12807921],
       [0.04824249, 0.00163602, 0.06636058, 0.02404066],
       [0.02978579, 0.00243829, 0.04654513, 0.09802434],
       [0.03546526, 0.00205597, 0.06296797, 0.02392207],
       [0.04762353, 0.00424887, 0.09144363, 0.12839478],
       [0.09441691, 0.0073859 , 0.16158233, 0.21939378],
       [0.01313927, 0.00073982, 0.02321099, 0.00734276],
       [0.05276531, 0.00772553, 0.10948858, 0.2388596 ],
       [0.08507317, 0.00476452, 0.12165361, 0.13278423],
       [0.04624102, 0.00676708, 0.11553867, 0.181785  ]])

## 5. Train

In [ ]:
lr = LinearRegression()
lr.fit(d_train, y_train, fit_intercept=True)

In [ ]:
lr.predict(d_test)

array([[ 0.18629184,  0.17006264, -0.12549452, -0.10206219],
       [ 0.16817211,  0.34450996, -0.03676555,  0.01653856],
       [ 0.18516346,  0.07271142, -0.06513045,  0.00121991],
       [ 0.19343302,  0.7717154 ,  0.14227777,  0.10724525],
       [ 0.84792425,  0.7923201 ,  0.05643331, -0.07499322],
       [ 0.16375114,  0.22141634,  0.01775145,  0.0033983 ],
       [ 0.71452632,  0.34157329,  0.41182727,  0.11521234],
       [ 0.80049983,  0.40475499,  0.03868916,  0.16632212],
       [ 0.07354957,  0.08697091,  0.03622844,  0.04090434],
       [ 0.55789895,  0.60620064,  0.0297374 , -0.0922422 ],
       [ 0.33070626,  0.23614812, -0.03201366, -0.00635284],
       [ 0.56724019,  0.04862761,  0.09328471, -0.0161131 ],
       [ 0.80876085,  0.80505035,  0.0011794 ,  0.00213471],
       [ 0.32643671,  0.76481772, -0.38610943,  0.00367606],
       [ 0.38905422,  0.77954679, -0.01221963,  0.00458491],
       [ 0.32920836,  0.4958166 , -0.04879477, -0.04046273],
       [ 0.41538521,  0.

In [ ]:
x_test_est = [np.array([[y[0], y[2] -1j * y[3]],[y[2] +1j * y[3], y[1]]]) for y in lr.predict(d_test)]

In [ ]:
[entanglement_entropy(x, [0,1]) for x in x_test_est]

[np.float64(0.72404983139738),
 np.float64(1.0881205628707535),
 np.float64(0.6975021954707875),
 np.float64(0.8783137810586331),
 np.float64(0.8744363666301317),
 np.float64(0.9724164386520064),
 np.float64(0.26482492072625186),
 np.float64(1.0395103638246406),
 np.float64(0.5140309867917268),
 np.float64(1.3179167043107087),
 np.float64(1.1759850520241293),
 np.float64(0.8202193373325666),
 np.float64(0.9812975210799726),
 np.float64(0.38633243360330893),
 np.float64(1.195056181361117),
 np.float64(1.3099119808018391),
 np.float64(0.8729483836216733),
 np.float64(1.2780438533294785),
 np.float64(1.3148525519197185),
 np.float64(0.8930694592373287),
 np.float64(0.946964959361885),
 np.float64(0.8873991073382328),
 np.float64(0.9330900435917582),
 np.float64(0.7771395402401633),
 np.float64(1.2876612834777448),
 np.float64(0.7898290469834053),
 np.float64(1.1964012298028845),
 np.float64(1.2674197073971936),
 np.float64(0.6704795312844123),
 np.float64(1.0264562495490965),
 np.float64(

In [ ]:
np.array([entanglement_entropy(x.matrix,[0,1]) for x in x_test]) - np.array([entanglement_entropy(x, [0,1]) for x in x_test_est])

array([ 4.15223411e-14,  1.50990331e-14,  3.78586051e-14, -5.44009282e-15,
       -1.84297022e-14,  2.00950367e-14, -2.89213098e-14, -2.48689958e-14,
        4.28546088e-14, -2.22044605e-15,  1.55431223e-14,  2.62012634e-14,
       -2.46469511e-14, -4.88498131e-15, -7.99360578e-15,  6.21724894e-15,
        1.57651669e-14, -1.06581410e-14,  6.66133815e-15,  2.14273044e-14,
       -3.59712260e-14, -1.62092562e-14,  1.01030295e-14, -4.91828800e-14,
       -9.99200722e-15, -4.30766534e-14, -1.64313008e-14,  1.15463195e-14,
        7.66053887e-15,  4.88498131e-15, -4.19664303e-14, -9.99200722e-15,
       -2.22044605e-14, -5.37347944e-14,  7.77156117e-15, -7.63833441e-14,
       -2.44249065e-14, -1.31006317e-14, -2.42028619e-14,  3.23074900e-14,
       -3.55271368e-15, -1.44328993e-14, -2.44249065e-15, -1.11022302e-14,
        8.43769499e-15, -3.55271368e-15, -3.66373598e-14,  5.10702591e-15,
       -2.30926389e-14,  1.02140518e-14,  4.19664303e-14,  5.06261699e-14,
        4.44089210e-15,  